# 五日反轉：中位排名退出與換手成本比較 v28

2026-10-16。目的：比較回到中位排名才退出與同頻率無保留控制、既有五日反轉。依公開研究規劃固定規則；全部既有歷史已看過，非未見樣本外。

沿用既有資料快取及修正後公司行動／均價引擎。使用者已同意在舊 kernel 離線後建立新 kernel、恢復必要狀態。不重新抓資料、不改排程、不提交交易。

In [ ]:
# Locate this competition repository; do not restart an existing kernel.
import sys as _repo_sys, importlib.util as _repo_importlib
from pathlib import Path as _PlainPath
_candidates = [_PlainPath.cwd(), *_PlainPath.cwd().parents]
_repo_root = next((p for p in _candidates if (p / '99_維護與備份' / '工具' / '研究路徑.py').exists()), None)
if _repo_root is None:
    raise RuntimeError('Open Jupyter from this repository root or its 04_Notebook directory.')
if 'research_layout' not in _repo_sys.modules:
    _spec = _repo_importlib.spec_from_file_location('research_layout', _repo_root / '99_維護與備份' / '工具' / '研究路徑.py')
    _mod = _repo_importlib.module_from_spec(_spec)
    _repo_sys.modules['research_layout'] = _mod
    _spec.loader.exec_module(_mod)
from research_layout import ResearchPath as Path
# New authorized kernel: inspect before restoring computational state.
%whos
import sys,os,json,gzip,hashlib,datetime,ast,collections
from research_layout import ResearchPath as Path
import numpy as np
import nbformat
print("V28_KERNEL",{"python":sys.executable,"pid":os.getpid(),"numpy":np.__version__})
print("V28_HISTORY",get_ipython().history_manager.session_number)


In [ ]:
# Restore only cached numerical data, saved engine definitions and old reference rows.
v28_root=Path("C:/Users/weiha/Desktop/文件/03_課程資料/115-1/資料探勘與應用/B_Final Project (0920~1221)/競賽資料")
v28_book=nbformat.read(v28_root/"動能持股_開賽前公告配置研究_2026-10-14.ipynb",as_version=4)
v28_basebook=nbformat.read(v28_root/"動能基準研究_24日回測_2026-10-13.ipynb",as_version=4)
with gzip.open(v28_root/"策略研究_原資料快照_v3_2026-10-14.json.gz","rt",encoding="utf-8") as f:
    raw_data=json.load(f)
v28_snapshot=hashlib.sha256(json.dumps(raw_data,default=str,ensure_ascii=False,separators=(",",":")).encode("utf-8")).hexdigest()
v28_v4=json.loads((v28_root/"策略研究_逐窗口結果_v4_2026-10-14.json").read_text(encoding="utf-8"))
assert v28_snapshot==v28_v4["raw_snapshot_sha256"]
for key in ["price","institution","dividend"]:
    for row in raw_data[key]:row[1]=datetime.date.fromisoformat(row[1])
COUNT=25;HORIZON=24;REBALANCE=5;LOT=1000;INVESTED=.9
INITIAL=1_000_000_000.;FEE=.001425;TAX=.003
exec(v28_basebook.cells[5].source,globals())
# Only original action flags and percentile helper, not its old experiment grid.
exec(v28_basebook.cells[8].source.split("scores={")[0],globals())
cash_div_v15=cash_div.copy();stock_div_v15=stock_div.copy()
signal_returns_v15=signal_returns.copy();bad_action_v15=bad_action.copy()
for action in v28_v4["action_corrections"]:
    day=datetime.date.fromisoformat(action["ex_date"]);sid=action["stock_id"]
    if day not in di or sid not in si:continue
    t,j=di[day],si[sid]
    cash_div_v15[t,j]=float(action["cash"]);stock_div_v15[t,j]=float(action["stock_ratio"])
    signal_returns_v15[t,j]=close[t,j]/float(action["reference"])-1;bad_action_v15[t,j]=False
v28_holidays=json.loads((v28_root/"A策略_休市除權息對齊查核_v7_2026-10-15.json").read_text(encoding="utf-8"))
for action in v28_holidays["actions_applied"]:
    t,j=di[datetime.date.fromisoformat(action["effective_price_session"])],si[action["stock_id"]]
    assert np.isclose(close[t-1,j],action["before"],atol=.001)
    cash_div_v15[t,j]=action["cash"];stock_div_v15[t,j]=action["stock_ratio"]
    signal_returns_v15[t,j]=close[t,j]/action["reference"]-1;bad_action_v15[t,j]=False
assert int(bad_action_v15.sum())==12
good_exec_v15=np.isfinite(execution)&(execution>0)&np.isfinite(volume)&(volume>0)&np.isfinite(turnover)&(turnover>0)&np.isfinite(close)&(close>0)
v28_v7cell=next(c.source for c in v28_book.cells if c.cell_type=="code" and c.source.startswith("# v7 fixed-method engine revalidation:"))
v28_score_source=v28_v7cell[v28_v7cell.index("scores_v7="):v28_v7cell.index("scores_v7['AH']")].replace("_v7","_v15")
exec(v28_score_source,globals())
v28_features=json.loads((v28_root/"五策略家族_資料與覆蓋查核_v15_2026-10-15.json").read_text(encoding="utf-8"))
for k,sig in [("MOM","A"),("REV","REV5")]:
    assert hashlib.sha256(scores_v15[sig].tobytes()).hexdigest()==v28_features["feature_sha256"][k]
# Restore original settlement and function definition only; no baseline batch rerun.
v28_base_source=v28_basebook.cells[10].source
for node in ast.parse(v28_base_source).body:
    if isinstance(node,ast.FunctionDef) and node.name=="settle":exec(ast.get_source_segment(v28_base_source,node),globals())
v28_simnode=next(n for n in ast.parse(v28_base_source).body if isinstance(n,ast.FunctionDef) and n.name=="simulate")
v28_legacy_source=ast.get_source_segment(v28_base_source,v28_simnode)
v28_replay_source=v28_legacy_source.replace("def simulate(","def replay_family_v28(")
for old,new in [("bad_action[t]","bad_action_v15[t]"),("cash_div[t]","cash_div_v15[t]"),("stock_div[t]","stock_div_v15[t]"),("good_exec[t]","good_exec_v15[t]"),("scores[strategy]","scores_v15[strategy]")]:
    v28_replay_source=v28_replay_source.replace(old,new)
exec(v28_replay_source,globals())
v28_factor=json.loads((v28_root/"等權非等權_四格控制逐窗口_v16_2026-10-15.json").read_text(encoding="utf-8"))
v28_old={k:v28_factor["results"][k] for k in ["REV_E0","MOM_E0"]}
v28_starts=np.arange(61,len(dates)-HORIZON+1)
v28_replay_checks=[]
for k,sig in [("REV_E0","REV5"),("MOM_E0","A")]:
    for ix in [0,100,200,400,649]:
        r=replay_family_v28(int(v28_starts[ix]),sig);o=v28_old[k][ix]
        assert r["invalid"]==o["invalid"] and r["violations"]==o["violations"]
        for field in ["return","mdd","cost","turnover","min_cash","max_cash"]:
            assert np.isclose(r[field],o[field],atol=1e-12,rtol=0),(k,ix,field)
        assert np.allclose(r["path"],o["path"],atol=1e-5,rtol=0)
        v28_replay_checks.append({"method":k,"index":ix,"matched":True})
print("V28_RESTORED",{"shape":shape,"dates":[str(dates[0]),str(dates[-1])],"windows":len(v28_starts),"snapshot":v28_snapshot,"unresolved_actions":int(bad_action_v15.sum()),"baseline_replay_checks":len(v28_replay_checks)})


In [ ]:
# Freeze implementation details before seeing the two new result sets.
v28_definition=json.loads((v28_root/"反轉中位退出_實驗定義_v28_2026-10-16.json").read_text(encoding="utf-8"))
v28_definition.update({"status":"registered_before_new_results","kernel_recovery_authorized":True,"kernel_connection":"kernel-aicup-reversal-20261016.json","kernel_pid":os.getpid(),"snapshot_sha256":v28_snapshot,"control_replay_checks":v28_replay_checks,"implementation_notes":{"missing_held_signal":"flag invalid and stop partial window, do not silently force exit","target25":"planned selected names and actual lot-positive shares must reach25 for candidate execution; actual20to30 separately official","daily_priority":"calculate exits/replacements then one target vector; equalweight on offsets0,5,10,15,20","replacements":"pretrade close market value per exiting stock; no redistribution of others on nonregular days","no_roundtrip":"one net delta per stock per session","trace":"retain daily execution/feasibility counts and planned decisions"}})
v28_definition_hash=hashlib.sha256(json.dumps(v28_definition,sort_keys=True,ensure_ascii=False).encode("utf-8")).hexdigest()
(v28_root/"反轉中位退出_實驗定義_v28_2026-10-16.json").write_text(json.dumps(v28_definition,ensure_ascii=False,indent=2),encoding="utf-8")
v28_r5=-scores_v15["REV5"]
v28_order=[];v28_medians=np.full(len(dates),np.nan)
for q in range(len(dates)):
    e=np.flatnonzero(np.isfinite(v28_r5[q]))
    order=e[np.lexsort((np.array(stock_ids)[e],v28_r5[q,e]))]
    v28_order.append(order)
    if len(e):v28_medians[q]=np.median(v28_r5[q,e])
v28_caps=np.array([.25 if s=="2330" else .10 for s in stock_ids])
def simulate_reversal_v28(start,mode,stress=0.):
    assert mode in ["median","daily25"]
    shares=np.zeros(len(stock_ids),dtype=np.int64);cash=INITIAL
    receivable_cash=0.;receivable_shares=np.zeros(len(stock_ids));values=[INITIAL]
    gross_traded=0.;fees=0.;invalid=False;violations=0;min_cash=1.;max_cash=0.
    cap_age=np.zeros(len(stock_ids),dtype=int);reasons=collections.Counter();trace=[]
    average_exposure=[];negative_days=0;official_count_days=0;cash_upper_days=0;cap_days=0
    replacement_count=0;replacement_days=0;initial_members=None;minimum_holdings=999
    for offset,t in enumerate(range(start,start+HORIZON)):
        before=shares.copy()
        if np.any(bad_action_v15[t]&((shares+receivable_shares)>0)):
            invalid=True;reasons["unknown_held_company_action"]+=1
        receivable_cash+=float(np.dot(before,cash_div_v15[t]))
        receivable_shares+=before*stock_div_v15[t]
        previous_nav=cash+np.dot(shares,np.nan_to_num(mark[t-1]))
        order=v28_order[t-1]
        if len(order)<COUNT:
            invalid=True;reasons["eligible_less25"]+=1;break
        held=np.flatnonzero(shares>0);regular=offset%REBALANCE==0
        target=shares.copy();exits=[];adds=[]
        if offset==0:
            selected=order[:COUNT].copy();initial_members=selected.tolist()
        else:
            if len(held)!=COUNT:
                invalid=True;reasons["actual_holdings_not_target25"]+=1;break
            if np.any(~np.isfinite(v28_r5[t-1,held])):
                invalid=True;reasons["missing_held_signal"]+=1;break
            if mode=="median":
                exits=[int(j) for j in held if v28_r5[t-1,j]>v28_medians[t-1]]
            else:
                top=set(order[:COUNT].tolist());exits=[int(j) for j in held if j not in top]
            exits=sorted(exits,key=lambda j:stock_ids[j])
            candidates=[int(j) for j in order if j not in set(held)]
            if len(candidates)<len(exits):
                invalid=True;reasons["replacement_shortage"]+=1;break
            adds=candidates[:len(exits)]
            selected=np.array([j for j in held if j not in set(exits)]+adds,dtype=int)
            if len(selected)!=COUNT:
                invalid=True;reasons["planned_names_not25"]+=1;break
            if not regular:
                needed=np.array(exits+adds,dtype=int)
                if len(needed) and np.any(~np.isfinite(close[t-1,needed])|(close[t-1,needed]<=0)):
                    invalid=True;reasons["missing_sizing_close"]+=1;break
                for old,new in zip(exits,adds):
                    budget=shares[old]*close[t-1,old]
                    target[old]=0
                    target[new]=int(np.floor(budget/close[t-1,new]/LOT))*LOT
        if regular:
            if np.any(~np.isfinite(close[t-1,selected])|(close[t-1,selected]<=0)):
                invalid=True;reasons["missing_sizing_close"]+=1;break
            target=np.zeros_like(shares)
            target[selected]=(np.floor((INVESTED/COUNT)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)
        if np.count_nonzero(target)!=COUNT:
            invalid=True;reasons["lot_positive_target_less25"]+=1;break
        delta=target-shares
        cost=0.;traded=0.;missing=False
        if np.any(delta):
            cash,shares,traded,cost,missing=settle(cash,shares,target,execution[t],good_exec_v15[t],fee=FEE+stress,tax=TAX)
            gross_traded+=traded;fees+=cost
            if missing:invalid=True;reasons["ordered_execution_unavailable"]+=1
        if exits:replacement_count+=len(exits);replacement_days+=1
        nav=cash+np.dot(shares,np.nan_to_num(mark[t]))
        aum=nav+receivable_cash+np.dot(receivable_shares,np.nan_to_num(mark[t]))
        if np.any((shares>0)&~np.isfinite(mark[t])):invalid=True;reasons["missing_valuation"]+=1
        weights=shares*np.nan_to_num(mark[t])/nav
        cap_age=np.where(weights>v28_caps+1e-12,cap_age+1,0)
        neg=cash<0;upper=cash/nav>=.25;count_bad=not 20<=np.count_nonzero(shares)<=30;cap_bad=np.any(cap_age>5)
        negative_days+=int(neg);cash_upper_days+=int(upper);official_count_days+=int(count_bad);cap_days+=int(cap_bad)
        violations+=int(neg or upper or count_bad or cap_bad)
        min_cash=min(min_cash,cash/nav);max_cash=max(max_cash,cash/nav)
        minimum_holdings=min(minimum_holdings,int(np.count_nonzero(shares)))
        average_exposure.append(float(weights.sum()));values.append(float(aum))
        trace.append({"offset":offset,"t":t,"regular":regular,"exit":exits,"add":adds,"traded":float(traded),"cost":float(cost),"cash":float(cash),"nav":float(nav),"aum":float(aum),"holdings":int(np.count_nonzero(shares)),"cash_ratio":float(cash/nav),"missing_execution":missing,"negative_cash":bool(neg),"count_violation":bool(count_bad),"cash_upper_violation":bool(upper),"cap_violation":bool(cap_bad)})
    vals=np.array(values)
    return {"start":start,"end":start+HORIZON-1,"return":float(vals[-1]/INITIAL-1),"mdd":float(np.min(vals/np.maximum.accumulate(vals)-1)),"turnover":float(gross_traded/INITIAL),"cost":float(fees/INITIAL),"invalid":bool(invalid),"violations":violations,"min_cash":float(min_cash),"max_cash":float(max_cash),"path":vals.tolist(),"completed_days":len(vals)-1,"invalid_reasons":dict(reasons),"negative_cash_days":negative_days,"cash_upper_days":cash_upper_days,"official_count_days":official_count_days,"cap_days":cap_days,"average_exposure":float(np.mean(average_exposure)) if average_exposure else None,"replacement_count":replacement_count,"replacement_days":replacement_days,"minimum_holdings":minimum_holdings if minimum_holdings<999 else None,"initial_members":initial_members,"daily":trace}
print("V28_FROZEN",v28_definition_hash,"new methods=2; fixed Bonferroni family10")


In [ ]:
# Execute the two preregistered candidates only; reuse archived controls.
import inspect,time
v28_definition["implementation_sha256"]=hashlib.sha256(inspect.getsource(simulate_reversal_v28).encode("utf-8")).hexdigest()
v28_definition_hash=hashlib.sha256(json.dumps(v28_definition,sort_keys=True,ensure_ascii=False).encode("utf-8")).hexdigest()
(v28_root/"反轉中位退出_實驗定義_v28_2026-10-16.json").write_text(json.dumps(v28_definition,ensure_ascii=False,indent=2),encoding="utf-8")
v28_probe={}
for ix in [0,100,200,400,649]:
    a=simulate_reversal_v28(int(v28_starts[ix]),"median")
    b=simulate_reversal_v28(int(v28_starts[ix]),"daily25")
    assert a["initial_members"]==b["initial_members"]
    assert np.isclose(a["path"][1],b["path"][1],atol=1e-5,rtol=0)
    assert np.isclose(a["path"][1],v28_old["REV_E0"][ix]["path"][1],atol=1e-5,rtol=0)
    v28_probe[ix]={"same_initial_members":True,"same_day1_nav":True}
v28_results={"B2":v28_old["REV_E0"],"B0":v28_old["MOM_E0"]}
v28_started=time.monotonic()
for key,mode in [("C1","median"),("C1_N","daily25")]:
    v28_results[key]=[]
    for ix,start in enumerate(v28_starts):
        v28_results[key].append(simulate_reversal_v28(int(start),mode))
        if (ix+1)%200==0:print("V28_PROGRESS",key,ix+1,flush=True)
    print("V28_DONE",key,{"n":len(v28_results[key]),"valid":sum(not r["invalid"] and r["violations"]==0 and r.get("completed_days",24)==24 for r in v28_results[key]),"elapsed_seconds":round(time.monotonic()-v28_started,1)},flush=True)
v28_export={"definition":v28_definition,"definition_sha256":v28_definition_hash,"probe":v28_probe,"reuse":{"B2":"v16 REV_E0; 5replays+feature/snapshot matched","B0":"v16 MOM_E0;5replays+feature/snapshot matched"},"new_results":{k:v28_results[k] for k in ["C1","C1_N"]}}
(v28_root/"反轉中位退出_逐窗口_v28_2026-10-16.json").write_text(json.dumps(v28_export,ensure_ascii=False,allow_nan=False),encoding="utf-8")
print("V28_SAVED_NEW_WINDOWS",1300,"no archived grids rerun")


In [ ]:
# Same-date net results and full-position moving-block uncertainty.
v28_keys=["B0","B2","C1_N","C1"]
v28_valid={k:np.array([not r["invalid"] and r["violations"]==0 and r.get("completed_days",24)==24 and np.isfinite(r["return"]) for r in v28_results[k]]) for k in v28_keys}
v28_array={f:{k:np.array([r[f] for r in v28_results[k]],float) for k in v28_keys} for f in ["return","mdd","cost","turnover"]}
v28_common=np.logical_and.reduce(list(v28_valid.values()))
v28_train=np.array([dates[int(s)+23]<=datetime.date(2024,12,31) for s in v28_starts])
v28_later=np.array([dates[int(s)]>=datetime.date(2025,1,1) for s in v28_starts])
def metrics_v28(key,mask):
    x=v28_array["return"][key][mask];m=v28_array["mdd"][key][mask]
    if not len(x):return {"n":0}
    wins=x[x>0];loss=x[x<0]
    result={"n":len(x),"win_rate":float(np.mean(x>0)),"flat":int(np.sum(x==0)),"expectancy":float(x.mean()),"median":float(np.median(x)),"mean_win":float(wins.mean()) if len(wins) else None,"mean_loss":float(loss.mean()) if len(loss) else None,"P10":float(np.quantile(x,.1)),"worst_return":float(x.min()),"median_mdd":float(np.median(m)),"worst_mdd":float(m.min()),"mdd_loss_P90":float(np.quantile(-m,.9)),"mean_cost":float(v28_array["cost"][key][mask].mean()),"mean_turnover_two_sided":float(v28_array["turnover"][key][mask].mean())}
    assert np.isclose(result["expectancy"],sum(wins)/len(x)+sum(loss)/len(x),atol=1e-12)
    if key in ["C1","C1_N"]:
        rows=[v28_results[key][i] for i in np.flatnonzero(mask)]
        result.update({"mean_exposure":float(np.mean([r["average_exposure"] for r in rows])),"mean_replacements":float(np.mean([r["replacement_count"] for r in rows])),"mean_replacement_days":float(np.mean([r["replacement_days"] for r in rows]))})
    return result
v28_common_metrics={k:metrics_v28(k,v28_common) for k in v28_keys}
v28_coverage={}
for k in v28_keys:
    rows=v28_results[k]
    reasons=collections.Counter()
    for r in rows:reasons.update(r.get("invalid_reasons",{}))
    v28_coverage[k]={"planned":650,"valid":int(v28_valid[k].sum()),"invalid":sum(bool(r["invalid"]) for r in rows),"violation_windows":sum(r["violations"]>0 for r in rows),"negative_cash_windows":sum(r.get("negative_cash_days",int(r["min_cash"]<0))>0 for r in rows),"partial_windows":sum(r.get("completed_days",len(r["path"])-1)<24 for r in rows),"reason_counts":dict(reasons),"own_valid_metrics":metrics_v28(k,v28_valid[k])}
v28_year_masks={str(y):np.array([dates[int(s)].year==y and dates[int(s)+23].year==y for s in v28_starts]) for y in [2024,2025,2026]}
v28_year_common={y:{k:metrics_v28(k,v28_common&mask) for k in v28_keys} for y,mask in v28_year_masks.items()}
v28_reps=20000;v28_rng=np.random.default_rng(20261016)
v28_boot={}
for block in [24,48]:
    n=len(v28_starts);num=int(np.ceil(n/block))
    v28_boot[block]=(v28_rng.integers(0,n-block+1,size=(v28_reps,num))[:,:,None]+np.arange(block)).reshape(v28_reps,-1)[:,:n]
def interval_v28(vector,mask,block,adjusted=True):
    z=np.where(mask,vector,np.nan)[v28_boot[block]]
    den=np.isfinite(z).sum(axis=1);ok=den>0
    means=np.nansum(z[ok],axis=1)/den[ok]
    alpha=.05/10 if adjusted else .05
    return {"limits":np.quantile(means,[alpha/2,1-alpha/2]).tolist(),"valid_reps":int(ok.sum()),"zero_denominator_reps":int((~ok).sum())}
v28_contrasts={}
for base in ["C1_N","B2"]:
    pair=v28_valid["C1"]&v28_valid[base];d=v28_array["return"]["C1"]-v28_array["return"][base]
    rec={"paired_n":int(pair.sum()),"base_valid_n":int(v28_valid[base].sum()),"coverage":float(pair.sum()/v28_valid[base].sum()),"mean_excess":float(d[pair].mean()),"beat_rate":float(np.mean(d[pair]>0)),"candidate":metrics_v28("C1",pair),"baseline":metrics_v28(base,pair),"cost_change":float((v28_array["cost"]["C1"]-v28_array["cost"][base])[pair].mean()),"turnover_change":float((v28_array["turnover"]["C1"]-v28_array["turnover"][base])[pair].mean())}
    for block in [24,48]:
        rec["block"+str(block)]=interval_v28(d,pair,block)
        # Descriptive 95% uncertainty in winning-window difference.
        wr=(v28_array["return"]["C1"]>0).astype(float)-(v28_array["return"][base]>0).astype(float)
        rec["win_rate_difference_block"+str(block)]=interval_v28(wr,pair,block,adjusted=False)
    rec["years"]={y:{"n":int((pair&m).sum()),"mean_excess":float(d[pair&m].mean()) if np.any(pair&m) else None} for y,m in v28_year_masks.items()}
    for label,section in [("training",v28_train),("later",v28_later)]:
        pm=pair&section;bm=v28_valid[base]&section
        rr={"n":int(pm.sum()),"base_n":int(bm.sum()),"coverage":float(pm.sum()/bm.sum()) if bm.any() else None,"mean_excess":float(d[pm].mean()) if pm.any() else None,"candidate":metrics_v28("C1",pm),"baseline":metrics_v28(base,pm)}
        for block in [24,48]:rr["block"+str(block)]=interval_v28(d,pm,block) if pm.any() else None
        rec[label]=rr
    cm=rec["candidate"];bm=rec["baseline"]
    rec["risk_pass"]=bool(cm["P10"]>=bm["P10"]-.01 and cm["median_mdd"]>=bm["median_mdd"]-.01 and cm["worst_mdd"]>=bm["worst_mdd"]-1e-12)
    rec["mean_pass"]=all(rec["block"+str(b)]["limits"][0]>0 for b in [24,48])
    rec["training_count_coverage_pass"]=bool(rec["training"]["n"]>=100 and rec["training"]["coverage"]>=.8)
    rec["coverage_pass"]=bool(rec["coverage"]>=.8)
    rec["numerical_gate_pass"]=all(rec[f] for f in ["risk_pass","mean_pass","training_count_coverage_pass","coverage_pass"])
    v28_contrasts[base]=rec
v28_phases={}
for phase in range(24):
    mask=(np.arange(650)%24==phase)&v28_valid["C1"]&v28_valid["B2"]
    v28_phases[str(phase)]={"n":int(mask.sum()),"candidate":metrics_v28("C1",mask),"baseline":metrics_v28("B2",mask),"mean_excess":float((v28_array["return"]["C1"]-v28_array["return"]["B2"])[mask].mean()) if mask.any() else None}
v28_summary={"definition_sha256":v28_definition_hash,"common_n":int(v28_common.sum()),"common_metrics":v28_common_metrics,"coverage":v28_coverage,"paired":v28_contrasts,"common_by_year":v28_year_common,"nonoverlap_phases":v28_phases,"bootstrap":{"reps":20000,"blocks":[24,48],"family":10,"seed":20261016,"full_position_axis":True},"no_untouched_holdout":True,"official_ActiveShare_verified":False}
print("V28_COMMON",json.dumps(v28_common_metrics,ensure_ascii=False))
print("V28_COVERAGE",json.dumps({k:{f:r[f] for f in ["valid","invalid","violation_windows","negative_cash_windows","partial_windows","reason_counts"]} for k,r in v28_coverage.items()},ensure_ascii=False))
print("V28_CONTRASTS",json.dumps({k:{f:r[f] for f in ["paired_n","coverage","mean_excess","cost_change","block24","block48","risk_pass","mean_pass","training_count_coverage_pass","numerical_gate_pass"]} for k,r in v28_contrasts.items()},ensure_ascii=False))
(v28_root/"反轉中位退出_統計與配對_v28_2026-10-16.json").write_text(json.dumps(v28_summary,ensure_ascii=False,allow_nan=False,indent=2),encoding="utf-8")


In [ ]:
# Fixed +10bp per side friction stress. Fee-equivalent diagnostic, not official fee or execution-price guarantee.
v28_stress_source=v28_replay_source.replace("def replay_family_v28(","def stress_replay_v28(")
line="settle(cash,shares,target,execution[t],good_exec_v15[t])"
assert v28_stress_source.count(line)==1
v28_stress_source=v28_stress_source.replace(line,"settle(cash,shares,target,execution[t],good_exec_v15[t],fee=FEE+.001,tax=TAX)")
exec(v28_stress_source,globals())
v28_stress={}
for key,mode in [("C1","median"),("C1_N","daily25"),("B2",None)]:
    v28_stress[key]=[simulate_reversal_v28(int(s),mode,stress=.001) if mode else stress_replay_v28(int(s),"REV5") for s in v28_starts]
    print("V28_STRESS_DONE",key,flush=True)
v28_stress_valid={k:np.array([not r["invalid"] and r["violations"]==0 and r.get("completed_days",24)==24 for r in rows]) for k,rows in v28_stress.items()}
v28_stress_summary={}
for base in ["C1_N","B2"]:
    keep=v28_stress_valid["C1"]&v28_stress_valid[base]&v28_valid["C1"]&v28_valid[base]
    ret={k:np.array([r["return"] for r in v28_stress[k]],float) for k in ["C1",base]}
    v28_stress_summary[base]={"n":int(keep.sum()),"candidate_mean":float(ret["C1"][keep].mean()),"base_mean":float(ret[base][keep].mean()),"mean_excess":float((ret["C1"]-ret[base])[keep].mean()),"original_cost_same_dates_excess":float((v28_array["return"]["C1"]-v28_array["return"][base])[keep].mean()),"candidate_negative_windows":sum(r.get("negative_cash_days",int(r["min_cash"]<0))>0 for r in v28_stress["C1"]),"base_negative_windows":sum(r.get("negative_cash_days",int(r["min_cash"]<0))>0 for r in v28_stress[base])}
v28_stress_export={"extra_fee_per_side":.001,"interpretation":"fee-equivalent additional trading friction; re-run quantities through NAV; not direct price slippage rewrite","results":v28_stress,"paired_summary":v28_stress_summary}
(v28_root/"反轉中位退出_成本壓力逐窗口_v28_2026-10-16.json").write_text(json.dumps(v28_stress_export,ensure_ascii=False,allow_nan=False),encoding="utf-8")
v28_summary["stress"]=v28_stress_summary
v28_summary["stress_coverage"]={k:{"valid":int(v28_stress_valid[k].sum()),"invalid":sum(r["invalid"] for r in rows),"violations":sum(r["violations"]>0 for r in rows)} for k,rows in v28_stress.items()}
print("V28_STRESS",json.dumps(v28_stress_summary))
print("V28_SPLITS",json.dumps({k:{section:{f:r[section][f] for f in ["n","base_n","coverage","mean_excess"]} for section in ["training","later"]} for k,r in v28_contrasts.items()}))


In [ ]:
# Persist stress arrays using explicit NumPy serialization; calculations from previous cell retained.
def serial_v28(value):
    if isinstance(value,np.ndarray):return value.tolist()
    if isinstance(value,np.generic):return value.item()
    if isinstance(value,(datetime.date,datetime.datetime)):return value.isoformat()
    raise TypeError(type(value).__name__)
(v28_root/"反轉中位退出_成本壓力逐窗口_v28_2026-10-16.json").write_text(json.dumps(v28_stress_export,ensure_ascii=False,allow_nan=False,default=serial_v28),encoding="utf-8")
v28_summary["stress"]=v28_stress_summary
v28_summary["stress_coverage"]={k:{"valid":int(v28_stress_valid[k].sum()),"invalid":int(sum(r["invalid"] for r in rows)),"violations":int(sum(r["violations"]>0 for r in rows))} for k,rows in v28_stress.items()}
print("V28_STRESS",json.dumps(v28_stress_summary,default=serial_v28))
print("V28_SPLITS",json.dumps({k:{section:{f:r[section][f] for f in ["n","base_n","coverage","mean_excess"]} for section in ["training","later"]} for k,r in v28_contrasts.items()},default=serial_v28))
# Audit the meaning of missing signal: lost quality qualification is not necessarily missing r5 price.
v28_quality={}
for key in ["C1","C1_N"]:
    cause=collections.Counter();cases=0
    for r in v28_results[key]:
        if "missing_held_signal" not in r["invalid_reasons"]:continue
        q=r["start"]+r["completed_days"]-1
        # Recover the final-day planned holding IDs by initial membership and executed successful replacements.
        held=set(r["initial_members"])
        for d in r["daily"]:
            if d["missing_execution"]:continue
            held.difference_update(d["exit"]);held.update(d["add"])
        for j in held:
            if np.isfinite(v28_r5[q,j]):continue
            cases+=1;rr=signal_returns_v15[q-59:q+1,j];ff=flow[q-19:q+1,j];vv=volume[q-19:q+1,j]
            if not np.all(np.isfinite(rr)&(rr>-1)):cause["60day_price_quality"]+=1
            if not np.all(np.isfinite(ff)&np.isfinite(vv)&(vv>0)):cause["20day_institution_or_volume_quality"]+=1
            if not np.isfinite(close[q,j]) or close[q,j]<=0:cause["current_close"]+=1
            if np.any(bad_action_v15[q-59:q+1,j]):cause["60day_unresolved_action_flag"]+=1
    v28_quality[key]={"stock_window_instances":cases,"overlapping_cause_counts":dict(cause),"scope":"research eligibility lost, not automatically official trading violation; repeated windows not unique events"}
v28_summary["lost_signal_quality"]=v28_quality
print("V28_LOST_QUALIFICATION",json.dumps(v28_quality))


In [ ]:
# Verification with invariant-instrumented replays of small selected cases; no new parameter variants.
v28_audit_source=inspect.getsource(simulate_reversal_v28).replace("def simulate_reversal_v28(","def audit_reversal_v28(")
v28_audit_source=v28_audit_source.replace("        delta=target-shares","        assert np.all(target>=0) and np.all(target%LOT==0)\n        if offset>0 and not regular:\n            untouched=np.ones(len(shares),dtype=bool);untouched[exits+adds]=False\n            assert np.array_equal(target[untouched],shares[untouched])\n            assert len(set(exits)&set(adds))==0\n        delta=target-shares")
exec(v28_audit_source,globals())
v28_invariant_checks=0
for key,mode in [("C1","median"),("C1_N","daily25")]:
    for ix in [0,100,200,400,649]:
        a=audit_reversal_v28(int(v28_starts[ix]),mode);b=v28_results[key][ix]
        for f in ["return","mdd","cost","turnover","min_cash","max_cash"]:assert np.isclose(a[f],b[f],atol=1e-12,rtol=0)
        assert a["invalid"]==b["invalid"] and a["violations"]==b["violations"]
        v28_invariant_checks+=1
    for ix,r in enumerate(v28_results[key]):
        assert r["start"]==int(v28_starts[ix]) and r["end"]==r["start"]+23
        assert len(r["path"])==r["completed_days"]+1==len(r["daily"])+1
        assert np.isclose(r["return"],r["path"][-1]/INITIAL-1,atol=1e-12)
        assert np.isclose(r["path"][1],v28_old["REV_E0"][ix]["path"][1],atol=1e-5,rtol=0)
        assert np.isclose(sum(d["cost"] for d in r["daily"])/INITIAL,r["cost"],atol=1e-12)
        assert np.isclose(sum(d["traded"] for d in r["daily"])/INITIAL,r["turnover"],atol=1e-12)
        for d in r["daily"]:
            assert d["t"]==r["start"]+d["offset"]
            assert d["holdings"]<=30
            if d["traded"]>0:assert FEE-1e-12<=d["cost"]/d["traded"]<=FEE+TAX+1e-12
            if mode=="median":
                assert all(v28_r5[d["t"]-1,j]>v28_medians[d["t"]-1] for j in d["exit"])
            assert not set(d["exit"])&set(d["add"])
v28_excluded=v28_valid["B2"]&~v28_valid["C1"]
v28_summary["baseline_dates_excluded_by_candidate"]={"n":int(v28_excluded.sum()),"B2":metrics_v28("B2",v28_excluded),"warning":"outcome diagnostic, not justification for excluding dates"}
v28_summary["verification"]={"baseline_sample_replays":10,"invariant_replays":v28_invariant_checks,"candidate_rows_checked":1300,"identical_first_day_to_B2_all_candidates":True,"path_fee_turnover_identities":True,"median_exit_uses_previous_session":True,"nonregular_untouched_shares_instrumented":True,"new_official_cost_definitions":2,"stress_definitions":3,"kernel_restarted_or_cleared":False,"kernel_connection":"kernel-aicup-reversal-20261016.json","ipython_history_session":get_ipython().history_manager.session_number,"notebook_full_restart_runall":False,"numpy_export_error_fixed_without_recompute":True}
(v28_root/"反轉中位退出_統計與配對_v28_2026-10-16.json").write_text(json.dumps(v28_summary,ensure_ascii=False,default=serial_v28,allow_nan=False,indent=2),encoding="utf-8")
print("V28_VERIFIED",v28_summary["verification"])
print("V28_PRIMARY_PAIR",json.dumps({k:{"candidate":r["candidate"],"baseline":r["baseline"],"years":r["years"]} for k,r in v28_contrasts.items()}))
print("V28_EXCLUDED_BASE",v28_summary["baseline_dates_excluded_by_candidate"])


In [ ]:
# Final Traditional Chinese report generated from verified kernel summaries, no hand-copied metrics.
labels28={"B0":"動能基準","B2":"既有五日反轉","C1_N":"每日跌出前25即換股","C1":"反轉＋中位排名退出"}
pct28=lambda v:"—" if v is None else f"{v*100:.2f}%"
pp28=lambda v:f"{v*100:+.3f}pp"
ci28=lambda c:"["+f"{c['limits'][0]*100:+.3f}, {c['limits'][1]*100:+.3f}"+"]pp"
lines28=[
"# 五日反轉＋中位排名退出：比較結果 v28",
"",
"日期：2026-10-16（台灣時間）。本次已完成研究比較；未提交交易，未變更原策略、排程或既有成果。",
"",
"## 結論",
"",
"**暫不取代既有五日反轉或動能基準。**中位排名退出明顯改善每日跌出前25就換股的過度換手控制；但對既有每五日換股反轉，平均增量未通過區塊校正不確定性與訓練覆蓋門檻。這不是證明所有保留機制都無效，也不以正均值強選新策略。",
"",
"## 已固定的三種反轉規則",
"",
"- B2：既有近五日調整後報酬由低至高挑25檔，每五日重選與等權；直接引用v16的REV_E0。",
"- C1：初始同樣25檔，之後每日只把近五日報酬嚴格高於當日合資格池中位數的持股換掉，補最低報酬且未持有者；同分依股票代碼。非例行日替補沿用被換出股的前日市值預算，其他股票股數不變。",
"- C1_N：與C1同樣每天檢查、每五日等權，但持股一跌出反轉前25就換掉，作為同頻率無保留機制控制。",
"- C1及C1_N在第1、6、11、16、21交易日，對換股後25檔恢復90%等權。當日只結算一個淨委託向量，無同股買回／賣出當沖。新、舊所有1300個候選窗口首日NAV均與B2一致。",
"",
"全部保留24日、90%目標投入、整千股、前一日資訊及前收盤換算、次日均價結算；手續費買賣各0.1425%、賣出另0.3%稅，應收股息／股票不作可用下單現金。期末估值，不強制清倉。25檔與90%只是本輪控制，官方持股下限是20而非25。",
"",
"來源：[de Groot等反轉成本研究接受稿](https://repub.eur.nl/pub/25718/AnotherLook_2011.pdf)；本次屬台股只做多與競賽配置改編，不是原論文美歐多空策略重現。來源提取及預先研究計畫見[公開教程研究](公開交易教程_24日競賽策略研究.md)。",
"",
"## 主要對照：C1與既有五日反轉的426個共同窗口",
"",
"|方法|窗口|費後勝率|24日期望值|平均獲利|平均虧損|P10|中位最大回撤|最差最大回撤|平均費稅|",
"|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
for k in ["B2","C1"]:
    m=v28_contrasts["B2"]["candidate" if k=="C1" else "baseline"]
    lines28.append("|"+labels28[k]+"|"+str(m["n"])+"|"+"|".join(pct28(m[f]) for f in ["win_rate","expectancy","mean_win","mean_loss","P10","median_mdd","worst_mdd","mean_cost"])+"|")
lines28 += [
"",
f"C1減B2費後均值 {pp28(v28_contrasts['B2']['mean_excess'])}；24日區塊校正區間 {ci28(v28_contrasts['B2']['block24'])}，48日 {ci28(v28_contrasts['B2']['block48'])}，均跨零。費稅反而增加 {pp28(v28_contrasts['B2']['cost_change'])}。保留排名比每日強制前25少交易，但不一定比原本每五日換股省費。",
"",
"勝率＝費後24日正報酬窗口比例；期望值＝算術平均24日報酬，含虧損。不是個股交易勝率，也不是競賽奪冠率。",
"",
"## 機制對照：C1與每日無保留控制的407個共同窗口",
"",
"|方法|費後勝率|24日期望值|平均費稅|雙邊換手／本金|平均換出股次|平均換股天數／24日|",
"|---|---:|---:|---:|---:|---:|---:|"]
for k in ["C1_N","C1"]:
    m=v28_contrasts["C1_N"]["candidate" if k=="C1" else "baseline"]
    lines28.append("|"+labels28[k]+"|"+"|".join(pct28(m[f]) for f in ["win_rate","expectancy","mean_cost","mean_turnover_two_sided"])+"|"+f"{m['mean_replacements']:.2f}|{m['mean_replacement_days']:.2f}|")
lines28 += [
"",
f"C1減每日控制的費後均值 {pp28(v28_contrasts['C1_N']['mean_excess'])}；24日區塊校正區間 {ci28(v28_contrasts['C1_N']['block24'])}，48日 {ci28(v28_contrasts['C1_N']['block48'])}。平均費稅降低 {pp28(-v28_contrasts['C1_N']['cost_change'])}。通過這一對照的數值門檻，不能據此宣稱優於既有B2；兩對照使用不同共同日期。",
"",
"中位退出仍平均約22天有換股，並未變成低頻持有策略。上述成本差是既有交易路徑的會計描述，不是把全部報酬差都歸因費用的因果估計；兩者選股及進出時間也不同。",
"",
"## 四方法完全相同的374個窗口",
"",
"|方法|費後勝率|24日期望值|最差最大回撤|平均費稅|",
"|---|---:|---:|---:|---:|"]
for k in v28_keys:
    m=v28_common_metrics[k]
    lines28.append("|"+labels28[k]+"|"+"|".join(pct28(m[f]) for f in ["win_rate","expectancy","worst_mdd","mean_cost"])+"|")
lines28 += [
"",
"動能在此共同表的均值仍高於C1；這是同日期描述，沒有新增或宣稱通過C1對動能的確認性統計檢定。不能與v15的313或v17的511窗口表交叉相減。",
"",
"## 可執行性與排除：不能只看成功子樣本",
"",
"|方法|計畫窗口|有效|資料／研究資格不可判定|曾官方限制違反|曾負現金|未完成24日|",
"|---|---:|---:|---:|---:|---:|---:|"]
for k in v28_keys:
    r=v28_coverage[k];lines28.append("|"+labels28[k]+"|"+"|".join(str(r[f]) for f in ["planned","valid","invalid","violation_windows","negative_cash_windows","partial_windows"])+"|")
lines28 += [
"",
"不可判定、違規與部分窗口可重疊，不能直接相加。C1本輪沒有觀察到負現金或已建模限制違反，不等於保證未來合規；Active Share歷史基準仍未驗證。",
"",
"C1有196個不可判定窗口：186曾因持股失去原共同訊號品質資格而提前停止；另有未知公司行動、委託成交不可用、實際持股未達本輪目標25等原因，可能重疊。缺訊號包含60日價格、20日法人／量、未知公司行動資格，不是全部都代表五日價格不存在。這些是研究品質排除，不自動等於官方違規。",
"",
"本輪目標25檔而不足25的研究排除，不代表官方20～30檔規定必然違反；沒有事後放寬規則補回樣本。每日檢查比五日檢查更容易遇到資格缺漏，這也是兩者覆蓋差異的一部分。",
"",
f"C1與B2共同426／B2有效531，覆蓋 {pct28(v28_contrasts['B2']['coverage'])}；B2有效但C1不可判定的105個窗口，B2均值 {pct28(v28_summary['baseline_dates_excluded_by_candidate']['B2']['expectancy'])}、最差回撤 {pct28(v28_summary['baseline_dates_excluded_by_candidate']['B2']['worst_mdd'])}。它們不能填0或當C1已避開風險。",
"",
"## 時間分段與預先門檻",
"",
"訓練僅用2024年底前已結束窗口；後續僅用2025年起始窗口，跨界窗口不納入兩者。資料以前已看過，這仍是時間分段回溯檢查，非真正未見樣本外。",
"",
"|C1相對對照|訓練配對／對照有效|訓練覆蓋|訓練均值增量|後續配對|後續均值增量|均值區間門檻|風險點估門檻|整體數值門檻|",
"|---|---:|---:|---:|---:|---:|---|---|---|"]
for b in ["C1_N","B2"]:
    r=v28_contrasts[b];tr=r["training"];la=r["later"]
    lines28.append("|"+labels28[b]+f"|{tr['n']}/{tr['base_n']}|"+pct28(tr["coverage"])+"|"+pp28(tr["mean_excess"])+f"|{la['n']}|"+pp28(la["mean_excess"])+"|"+("通過" if r["mean_pass"] else "未通過")+"|"+("通過" if r["risk_pass"] else "未通過")+"|"+("通過此對照" if r["numerical_gate_pass"] else "未通過")+"|")
lines28 += [
"",
"完整採用需對兩個主要對照通過；C1對B2訓練有112窗口但只覆蓋62.2%，未達80%，且均值區間跨零，因此不升格。風險門檻為P10及中位MDD不得惡化逾1pp、最差MDD不得更差；點估過關不代表風險保證。",
"",
"|起迄同年：C1減B2|配對窗口|費後均值增量|",
"|---|---:|---:|"]
for y,r in v28_contrasts["B2"]["years"].items():lines28.append("|"+y+"|"+str(r["n"])+"|"+pp28(r["mean_excess"])+"|")
lines28 += [
"",
"2024為負增量，2025與2026為正，未形成一致優勢。跨年窗口保留在總表，因此逐年表筆數不等於426。",
"",
"區間採完整650個起始日位置的24／48日移動區塊bootstrap，各20,000次，固定seed20261016、同批共同抽樣，保留缺漏位置；主要均值比較保留原規劃家族10個名額、Bonferroni校正，沒有因本次只跑兩比較而縮小家族。勝率差另存描述性95%區間，不冒充家族校正結論。20000次在每尾約50個抽樣點，仍有模擬與區塊長度假設限制。",
"",
"24個不重疊起始相位均保存，未挑最好相位。相位之間及滑動窗口仍相關，不是650場獨立競賽；本輪校正也不抹除前面多輪策略挑選偏誤。",
"",
"## 單邊額外10bp成本壓力",
"",
"以每次交易額外0.1%費用模擬不利交易摩擦，重新推進NAV、現金與股數，非只從最後報酬扣一筆。它不是官方另收費，也不是重寫成交價或保證實際滑價上限。",
"",
"|C1相對對照|同日期窗口|壓力後C1均值|壓力後對照均值|壓力後增量|",
"|---|---:|---:|---:|---:|"]
for b,r in v28_stress_summary.items():lines28.append("|"+labels28[b]+"|"+str(r["n"])+"|"+pct28(r["candidate_mean"])+"|"+pct28(r["base_mean"])+"|"+pp28(r["mean_excess"])+"|")
lines28 += [
"",
"壓力下對B2仍有正均值，但本表沒有新增顯著性宣稱，不能補救主比較未通過的區間及覆蓋門檻。",
"",
"## 恢復、驗證與限制",
"",
"- 舊兩個競賽kernel連線埠均無回應、Windows無Python／Jupyter程序；使用者明確同意後建立專用kernel。先執行%whos確認空狀態，Python為C:/Python314/python.exe，kernel PID "+str(os.getpid())+"，IPython history session "+str(get_ipython().history_manager.session_number)+"。",
"- 恢復快取734日×150檔，2023-10-02～2026-10-12，650起點；snapshot SHA256="+v28_snapshot+"。與既有動能／反轉特徵雜湊一致；10個舊控制抽查報酬、成本、換手、回撤、現金及資產路徑一致。原控制整批未重跑。",
"- 新官方費用定義2×650=1300窗口；另3×650成本壓力窗口。全1300列驗證首日一致、成本／換手及NAV終值恆等式、時序、退出條件。10個加入斷言的抽查確認非例行日未退出持股股數不變、整千股、非負股數及無同股往返。",
"- 首次成本壓力匯出遇NumPy型別JSON序列化錯誤；在同一kernel修復序列化後保存，沒有重算壓力實驗或覆寫舊研究。原錯誤與修復保留在Notebook。",
"- kernel保持運行，沒有restart／clear／shutdown，也未執行Restart Kernel and Run All。Notebook保存本次輸入及輸出；IPython history與kernel記憶體不是永久Notebook的替代。",
"- 現況150檔股票池具有存活／選擇偏誤；沿用近期原研究資格，不是歷史官方股票池或完整掛牌資料重建。未知公司行動、缺成交及價格、Active Share與正式DataPack未驗證仍限制外推。v27的十年價量資料不與本輪短期含法人資格快照混合。",
"",
"## 成果與下一步",
"",
"- [已執行Notebook](反轉中位退出_比較研究_v28_2026-10-16.ipynb)",
"- [事前定義與實作雜湊](反轉中位退出_實驗定義_v28_2026-10-16.json)",
"- [兩新方法完整窗口及日軌跡](反轉中位退出_逐窗口_v28_2026-10-16.json)",
"- [統計、分段、相位與驗證](反轉中位退出_統計與配對_v28_2026-10-16.json)",
"- [成本壓力完整窗口](反轉中位退出_成本壓力逐窗口_v28_2026-10-16.json)",
"",
"保留此結果作為反轉退出研究，不自行修改中位數、5日、持股數或投入比例直到顯著。下一候選RSI(2)尚未執行；本次授權只跑C1。研究結果不構成改採新交易策略，未新建Project、Git repository或對話。"
]
v28_report="\n".join(lines28)+"\n"
(v28_root/"反轉中位退出_比較報告_v28_2026-10-16.md").write_text(v28_report,encoding="utf-8")
v28_state_path=v28_root/"A策略_無人值守研究狀態.json"
v28_state=json.loads(v28_state_path.read_text(encoding="utf-8"))
v28_state["reversal_median_exit_v28"]={"status":"completed","running":False,"completed_at":datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=8))).isoformat(),"kernel_connection":str(Path(os.environ["APPDATA"])/"jupyter/runtime/kernel-aicup-reversal-20261016.json"),"kernel_pid":os.getpid(),"definition_sha256":v28_definition_hash,"new_official_windows":1300,"stress_windows":1950,"comparison":"C1 medianexit vs daily no buffer and archived B2","common_n":374,"paired_B2_n":426,"paired_B2_mean_excess":v28_contrasts["B2"]["mean_excess"],"passes_both_controls":False,"conclusion":"no replacement of existing reversal or momentum; daily buffer mechanism positive vs daily highturnover only","outputs":["反轉中位退出_比較研究_v28_2026-10-16.ipynb","反轉中位退出_比較報告_v28_2026-10-16.md","反轉中位退出_實驗定義_v28_2026-10-16.json","反轉中位退出_逐窗口_v28_2026-10-16.json","反轉中位退出_統計與配對_v28_2026-10-16.json","反轉中位退出_成本壓力逐窗口_v28_2026-10-16.json"],"verification":v28_summary["verification"]}
v28_state_path.write_text(json.dumps(v28_state,ensure_ascii=False,default=serial_v28,indent=2),encoding="utf-8")
print("V28_COMPLETE","report saved; no adopted strategy change; kernel retained")


## 本次結論

已完成兩個新方法、既有控制引用、區塊不確定性、成本壓力與帳務驗證。中位排名退出改善每日無緩衝控制，但尚未證實優於原每五日反轉；不取代既有策略。完整數值與限制見同目錄v28 Markdown報告與統計JSON。

新專用kernel保留；未Restart and Run All，未提交交易。